# 08 Beta Diversity and Settlement

We test whether fungal community composition is associated with settlement type (`Settlement_Type`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted model tests settlement type after accounting for age and diet.

## 1. Setup

### 1.1 Imports

In [1]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

from itertools import combinations  # To create settlement pairs for pairwise PERMANOVA

### 1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [4]:
# 1. Shared inputs and output directories
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/08_beta_diversity_settlement"
vis_dir = "../data/processed/08_beta_diversity_settlement"

columns = ["Settlement_Type", "age_years_sample", "diet_type_sample"]


# 2. Complete-case inputs for both models
adjusted_settlement_out_dir = f"{out_dir}/adjusted_settlement"

adjusted_settlement_distance_path = f"{adjusted_settlement_out_dir}/asv_bray_curtis_complete.qza"
kmer16_complete_distance_path = f"{adjusted_settlement_out_dir}/kmer16_bray_curtis_complete.qza"
adjusted_settlement_metadata_path = f"{adjusted_settlement_out_dir}/metadata_complete.tsv"

asv_complete_tsv_path = f"{adjusted_settlement_out_dir}/asv_bray_curtis_complete.tsv"
kmer16_complete_tsv_path = f"{adjusted_settlement_out_dir}/kmer16_bray_curtis_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"
asv_unadjusted_overall_path = f"{unadjusted_permanova_dir}/asv_overall.tsv"
kmer16_unadjusted_overall_path = f"{unadjusted_permanova_dir}/kmer16_overall.tsv"

asv_adjusted_overall_qzv = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_qzv = f"{adjusted_permanova_dir}/kmer16_overall.qzv"
asv_adjusted_overall_path = f"{adjusted_permanova_dir}/asv_overall.tsv"
kmer16_adjusted_overall_path = f"{adjusted_permanova_dir}/kmer16_overall.tsv"


# 4. Pairwise PERMANOVA inputs and outputs
pairwise_input_dir = f"{adjusted_settlement_out_dir}/pairwise_inputs"

asv_unadjusted_pairwise_path = f"{unadjusted_permanova_dir}/asv_pairwise.tsv"
kmer16_unadjusted_pairwise_path = f"{unadjusted_permanova_dir}/kmer16_pairwise.tsv"
asv_adjusted_pairwise_path = f"{adjusted_permanova_dir}/asv_pairwise.tsv"
kmer16_adjusted_pairwise_path = f"{adjusted_permanova_dir}/kmer16_pairwise.tsv"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_settlement_out_dir"
!mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"
!mkdir -p "$pairwise_input_dir"

### 1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [6]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

print("Whole-ASV samples:", len(asv_matrix.ids))
print("16-mer samples:", len(distance_matrix.ids))
print("Metadata rows:", len(metadata))

Whole-ASV samples: 150
16-mer samples: 150
Metadata rows: 150


## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [7]:
shared_ids = pd.Index(distance_matrix.ids).intersection(asv_matrix.ids)
shared_ids = shared_ids.intersection(metadata.index)

matched = metadata.loc[shared_ids, columns].copy()

print("Matched samples:", len(matched))

Matched samples: 150


### 2.2 Keep complete cases

We keep samples labeled Rural, Suburban, or Urban with a numeric age and a recorded diet type. Both PERMANOVA models use this same subset.

In [8]:
matched["Settlement_Type"] = matched["Settlement_Type"].str.strip()
matched["diet_type_sample"] = matched["diet_type_sample"].str.strip().replace("", pd.NA)
matched["age_years_sample"] = pd.to_numeric(matched["age_years_sample"], errors="coerce")

keep = matched["Settlement_Type"].isin(["Rural", "Suburban", "Urban"])
settlement_complete = matched.loc[keep].dropna(subset=columns)
complete_ids = settlement_complete.index.tolist()

print("Complete-case samples:", len(complete_ids))

Complete-case samples: 139


We subset both Bray–Curtis matrices to the complete-case sample IDs in the same order and save them with the matching metadata.

In [10]:
distance_complete = asv_matrix.filter(complete_ids)
kmer16_complete = distance_matrix.filter(complete_ids)

assert list(distance_complete.ids) == complete_ids
assert list(kmer16_complete.ids) == complete_ids

settlement_complete.index.name = "sample-id"
q2.Metadata(settlement_complete).save(adjusted_settlement_metadata_path)

q2.Artifact.import_data("DistanceMatrix", distance_complete).save(
    adjusted_settlement_distance_path
)
q2.Artifact.import_data("DistanceMatrix", kmer16_complete).save(
    kmer16_complete_distance_path
)
q2.Metadata(settlement_complete).save(adjusted_settlement_metadata_path)

print("Saved both distance matrices and metadata for", len(complete_ids), "samples.")

Saved both distance matrices and metadata for 139 samples.


### 2.3 Summarize samples

We compare settlement group sizes before and after requiring complete metadata. We also check how the diet groups are distributed across settlement types.

In [11]:
settlement_samples = matched.loc[keep]

counts = settlement_samples["Settlement_Type"].value_counts().to_frame("With settlement label")
counts["With all three variables"] = (
    settlement_complete["Settlement_Type"].value_counts().reindex(counts.index, fill_value=0)
)

display(counts)
display(pd.crosstab(settlement_complete["diet_type_sample"], settlement_complete["Settlement_Type"]))
print("Complete samples:", len(settlement_complete))

,With settlement label,With all three variables
Settlement_Type,,
Rural,65,61
Urban,56,54
Suburban,24,24


Settlement_Type,Rural,Suburban,Urban
diet_type_sample,,,
Omnivore,49,20,44
Omnivore but do not eat red meat,2,2,3
Vegan,3,0,2
Vegetarian,4,1,1
Vegetarian but eat seafood,3,1,4


Complete samples: 139


### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across the three settlement types. Both tests use the same 139 samples and include settlement type as the only predictor.

#### 3.1.1 Whole ASVs

In [12]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_settlement_distance_path" \
    --m-metadata-file "$adjusted_settlement_metadata_path" \
    --p-formula "Settlement_Type" \
    --o-visualization "$asv_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_unadjusted_overall_qzv)

#### 3.1.2 16-mers

In [13]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_settlement_metadata_path" \
    --p-formula "Settlement_Type" \
    --o-visualization "$kmer16_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_unadjusted_overall_qzv)

#### Unadjusted results

| **Settlement term** | **Whole ASVs** | **16-mers** |
| ------------------- | -------------- | ----------- |
| Samples             | 139            | 139         |
| R²                  | 0.0243         | 0.0223      |
| Pseudo-F            | 1.693          | 1.553       |
| p-value             | **0.021**          | 0.083       |

Settlement type explained a similar proportion of variation in both tests (2.43% for whole ASVs and 2.23% for 16-mers). Only the whole-ASV test was statistically significant at the 0.05 threshold.

### 3.2 Adjusted PERMANOVA

We put settlement type last in the formula to test whether fungal community composition differs among the three settlement types after accounting for age and diet. Both tests use the same 139 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [14]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_settlement_distance_path" \
    --m-metadata-file "$adjusted_settlement_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + Settlement_Type" \
    --o-visualization "$asv_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_adjusted_overall_qzv)

#### 3.2.2 16-mers

In [15]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_settlement_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + Settlement_Type" \
    --o-visualization "$kmer16_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_adjusted_overall_qzv)

#### Adjusted results

| **Settlement term** | **Whole ASVs** | **16-mers** |
| ------------------- | -------------- | ----------- |
| Samples             | 139            | 139         |
| R²                  | 0.0241         | 0.0221      |
| Pseudo-F            | 1.677          | 1.534       |
| p-value             | **0.022**          | 0.074       |

After accounting for age and diet, the whole-ASV test was statistically significant at the 0.05 threshold (p = 0.022). The 16-mer test did not reach that threshold (p = 0.074).

### 3.3 Compare unadjusted and adjusted overall PERMANOVA

All four tests used the same 139 samples. We compare the settlement term before and after accounting for age and diet.

| **Bray–Curtis representation** | **Model**  | **Settlement R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ----------------- | ------------ | ----------- |
| Whole ASVs                     | Unadjusted | 0.0243            | 1.693        | **0.021**       |
| Whole ASVs                     | Adjusted   | 0.0241            | 1.677        | **0.022**       |
| 16-mers                        | Unadjusted | 0.0223            | 1.553        | 0.083       |
| 16-mers                        | Adjusted   | 0.0221            | 1.534        | 0.074       |

The observed settlement R² decreased slightly after adjustment for both representations. The observed settlement R² was slightly higher for whole ASVs than for 16-mers in both models. At the 0.05 threshold, both whole-ASV tests were statistically significant, and both 16-mer tests had p-values above 0.05.

## 4. Pairwise PERMANOVA

We compare all three pairs of settlement types on the whole-ASV and 16-mer Bray–Curtis matrices. For each pair, we use the same samples in the unadjusted and adjusted tests. We apply Holm correction across the three comparisons separately for each representation and model.

### Prepare the pairwise inputs

We start from the 139 complete cases. For each settlement pair, we save a matching metadata file and subset both distance matrices to those samples.

In [16]:
settlement_pairs = list(combinations(sorted(settlement_complete["Settlement_Type"].unique()), 2))

for pair_number, (group_a, group_b) in enumerate(settlement_pairs, start=1):
    pair_metadata = settlement_complete.loc[
        settlement_complete["Settlement_Type"].isin([group_a, group_b])
    ]
    pair_ids = pair_metadata.index.tolist()

    Metadata(pair_metadata).save(f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv")

    asv_pair = distance_complete.filter(pair_ids)
    kmer16_pair = kmer16_complete.filter(pair_ids)

    Artifact.import_data("DistanceMatrix", asv_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    )
    Artifact.import_data("DistanceMatrix", kmer16_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    )

    print(f"{pair_number}. {group_a} vs. {group_b} (n = {len(pair_ids)})")

1. Rural vs. Suburban (n = 85)
2. Rural vs. Urban (n = 115)
3. Suburban vs. Urban (n = 78)


### 4.1 Unadjusted pairwise PERMANOVA

#### 4.1.1 Whole ASVs

We run one PERMANOVA per settlement pair using the whole-ASV Bray–Curtis distances.

In [18]:
for pair_number, (group_a, group_b) in enumerate(settlement_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{unadjusted_permanova_dir}/asv_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "Settlement_Type" \
        --o-visualization "$pair_qzv_path"

1. Rural vs. Suburban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/asv_pair_01.qzv
2. Rural vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/asv_pair_02.qzv
3. Suburban vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/asv_pair_03.qzv


In [19]:
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_01.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_02.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_03.qzv")

#### Unadjusted pairwise results · Whole ASVs

| Settlement pair | n | R² | Pseudo-F | Raw p | Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: | ---: |
| Rural vs. Suburban | 85 | 0.0242 | 2.062 | 0.036 | 0.072 |
| Rural vs. Urban | 115 | 0.0067 | 0.767 | 0.722 | 0.722 |
| Suburban vs. Urban | 78 | 0.0347 | 2.731 | 0.007 | **0.021** |

After Holm correction across the three comparisons, only Suburban vs. Urban was statistically significant at the 0.05 threshold (Holm-adjusted p = 0.021).

#### 4.1.2 16-mers

We repeat the three unadjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same samples for each settlement pair. Settlement is the only predictor.

In [20]:
for pair_number, (group_a, group_b) in enumerate(settlement_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{unadjusted_permanova_dir}/kmer16_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "Settlement_Type" \
        --o-visualization "$pair_qzv_path"

1. Rural vs. Suburban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/kmer16_pair_01.qzv
2. Rural vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/kmer16_pair_02.qzv
3. Suburban vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/unadjusted_permanova/kmer16_pair_03.qzv


In [21]:
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_01.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_02.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_03.qzv")

#### Unadjusted pairwise results · 16-mers

| **Settlement pair** | **Samples** | **R²** | **Pseudo-F** | **p-value** | **Holm-adjusted p** |
| --- | ---: | ---: | ---: | ---: | ---: |
| 1. Rural vs. Suburban | 85 | 0.0197 | 1.669 | 0.103 | 0.206 |
| 2. Rural vs. Urban | 115 | 0.0076 | 0.860 | 0.538 | 0.538 |
| 3. Suburban vs. Urban | 78 | 0.0317 | 2.490 | 0.036 | 0.108 |

None of the three pairwise comparisons was statistically significant after Holm correction.

#### Unadjusted results

| **Settlement pair** | **Whole-ASV R²** | **16-mer R²** | **Whole-ASV Holm-adjusted p** | **16-mer Holm-adjusted p** |
| --- | ---: | ---: | ---: | ---: |
| Rural vs. Suburban (n = 85) | 0.0242 | 0.0197 | 0.072 | 0.206 |
| Rural vs. Urban (n = 115) | 0.0067 | 0.0076 | 0.722 | 0.538 |
| Suburban vs. Urban (n = 78) | 0.0347 | 0.0317 | **0.021** | 0.108 |

Suburban vs. Urban had the highest observed R² for both representations. After Holm correction, this was the only statistically significant comparison at the 0.05 threshold, and it was significant for whole ASVs only.

### 4.2 Adjusted pairwise PERMANOVA

We reuse the three settlement pairs and sample subsets from section 4.1. For each pair, we test the settlement term after accounting for age and diet. We apply Holm correction across the three comparisons separately for whole ASVs and 16-mers.

#### 4.2.1 Whole ASVs

In [22]:
for pair_number, (group_a, group_b) in enumerate(settlement_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{adjusted_permanova_dir}/asv_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + diet_type_sample + Settlement_Type" \
        --o-visualization "$pair_qzv_path"

1. Rural vs. Suburban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/asv_pair_01.qzv
2. Rural vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/asv_pair_02.qzv
3. Suburban vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/asv_pair_03.qzv


In [23]:
# Visualization.load(f"{adjusted_permanova_dir}/asv_pair_01.qzv")
# Visualization.load(f"{adjusted_permanova_dir}/asv_pair_02.qzv")
# Visualization.load(f"{adjusted_permanova_dir}/asv_pair_03.qzv")

#### Adjusted pairwise results · Whole ASVs

| **Settlement pair** | **Samples** | **R²** | **Pseudo-F** | **Raw p** | **Holm-adjusted p** |
| --- | ---: | ---: | ---: | ---: | ---: |
| 1. Rural vs. Suburban | 85 | 0.0228 | 1.930 | 0.042 | 0.084 |
| 2. Rural vs. Urban | 115 | 0.0074 | 0.848 | 0.585 | 0.585 |
| 3. Suburban vs. Urban | 78 | 0.0351 | 2.716 | 0.009 | **0.027**|

After accounting for age and diet, Suburban vs. Urban remained the only whole-ASV pairwise comparison that was statistically significant after Holm correction (adjusted p = 0.027).

#### 4.2.2 16-mers

We repeat the ten adjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same sample subsets as in 4.2.1. We test the diet term after accounting for age and settlement type.

In [24]:
for pair_number, (group_a, group_b) in enumerate(settlement_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{adjusted_permanova_dir}/kmer16_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + diet_type_sample + Settlement_Type" \
        --o-visualization "$pair_qzv_path"

1. Rural vs. Suburban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/kmer16_pair_01.qzv
2. Rural vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/kmer16_pair_02.qzv
3. Suburban vs. Urban
Saved Visualization to: ../data/processed/08_beta_diversity_settlement/adjusted_permanova/kmer16_pair_03.qzv


In [25]:
# Visualization.load(f"{adjusted_permanova_dir}/kmer16_pair_01.qzv")
# Visualization.load(f"{adjusted_permanova_dir}/kmer16_pair_02.qzv")
# Visualization.load(f"{adjusted_permanova_dir}/kmer16_pair_03.qzv")

#### Adjusted pairwise results · 16-mers

| **Settlement pair** | **Samples** | **R²** | **Pseudo-F** | **Raw p** | **Holm-adjusted p** |
| --- | ---: | ---: | ---: | ---: | ---: |
| 1. Rural vs. Suburban | 85 | 0.0189 | 1.590 | 0.121 | 0.242 |
| 2. Rural vs. Urban | 115 | 0.0082 | 0.945 | 0.463 | 0.463 |
| 3. Suburban vs. Urban | 78 | 0.0311 | 2.392 | 0.027 | 0.081 |

After accounting for age and diet, none of the three 16-mer pairwise comparisons was statistically significant after Holm correction. Suburban vs. Urban had the highest observed R² and the lowest Holm-adjusted p-value (0.081).

#### Adjusted results

| **Settlement pair** | **Whole-ASV R²** | **16-mer R²** | **Whole-ASV Holm-adjusted p** | **16-mer Holm-adjusted p** |
| --- | ---: | ---: | ---: | ---: |
| Rural vs. Suburban (n = 85) | 0.0228 | 0.0189 | 0.084 | 0.242 |
| Rural vs. Urban (n = 115) | 0.0074 | 0.0082 | 0.585 | 0.463 |
| Suburban vs. Urban (n = 78) | 0.0351 | 0.0311 | **0.027** | 0.081 |

After accounting for age and diet, Suburban vs. Urban had the highest observed settlement R² and the lowest Holm-adjusted p-value for both representations. Only the whole-ASV test for this pair was statistically significant after Holm correction (adjusted p = 0.027).

Whole-ASV R² was higher in two of the three comparisons. The differences in R² between representations are descriptive.

### 4.3 Compare unadjusted and adjusted pairwise PERMANOVA

Both models used the same samples for each settlement pair. We compare the observed settlement R² and Holm-adjusted p-values before and after accounting for age and diet.

| **Settlement pair** | **Representation** | **Settlement R² (unadjusted → adjusted)** | **Holm-adjusted p (unadjusted → adjusted)** |
| --- | --- | ---: | ---: |
| Rural vs. Suburban (n = 85) | Whole ASVs | 0.0242 → 0.0228 | 0.072 → 0.084 |
| Rural vs. Suburban (n = 85) | 16-mers | 0.0197 → 0.0189 | 0.206 → 0.242 |
| Rural vs. Urban (n = 115) | Whole ASVs | 0.0067 → 0.0074 | 0.722 → 0.585 |
| Rural vs. Urban (n = 115) | 16-mers | 0.0076 → 0.0082 | 0.538 → 0.463 |
| Suburban vs. Urban (n = 78) | Whole ASVs | 0.0347 → 0.0351 | **0.021 → 0.027** |
| Suburban vs. Urban (n = 78) | 16-mers | 0.0317 → 0.0311 | 0.108 → 0.081 |

Suburban vs. Urban had the highest observed settlement R² for both representations in both models. Its R² changed little after accounting for age and diet. This pair was statistically significant after Holm correction in both whole-ASV models. All 16-mer comparisons had Holm-adjusted p-values above 0.05 in both models.

## 5. Discussion of overall and pairwise PERMANOVA

The overall whole-ASV tests provided evidence of an association between settlement type and fungal community composition, with and without adjustment for age and diet (p = 0.021 and 0.022, respectively). Both overall 16-mer tests had p-values above 0.05. After Holm correction, Suburban vs. Urban was the only statistically significant pairwise comparison, with whole-ASV adjusted p-values of 0.021 in the unadjusted model and 0.027 in the model accounting for age and diet. All 16-mer pairwise comparisons had Holm-adjusted p-values above 0.05.

The overall settlement R² ranged from 2.21% to 2.43% and changed little after accounting for age and diet. Suburban vs. Urban had the highest observed pairwise R² for both representations in both models (3.11% to 3.51%, n = 78). Each pairwise R² describes variation within its own two-group subset, so it answers a different question from the overall R² across all three settlement types.

Across the beta diversity analyses, we noticed that comparisons with higher R² values often had lower p-values. This makes sense because a higher R² means that the variable we tested accounted for a larger share of the differences in fungal community composition among samples. A difference between groups is easier to detect when it is large compared with how much the samples vary within each group. It is then less likely that randomly shuffling the samples would produce a result as extreme as the one we observed.